# Titanic Survival Prediction — Classification Models

## Setup: Download and Load the Dataset

In [ ]:
import urllib.request, zipfile, os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%204/Day%203/Titanic%20Dataset.zip"
urllib.request.urlretrieve(url, "titanic.zip")

with zipfile.ZipFile("titanic.zip", "r") as z:
    z.extractall("titanic_data")

csv_files = glob.glob("titanic_data/**/*.csv", recursive=True)
print("CSV files found:", csv_files)

In [ ]:
# Load train file (use the one containing 'Survived')
dfs = {os.path.basename(f): pd.read_csv(f) for f in csv_files}
for name, d in dfs.items():
    print(f"{name}: {d.shape}, columns: {d.columns.tolist()}")

In [ ]:
# Select the file that has the 'Survived' target column
df = next(d for d in dfs.values() if "Survived" in d.columns)
print("Dataset shape:", df.shape)
df.head()

## Exercise 1: Exploratory Data Analysis

In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
print(df.describe())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Survival count
df["Survived"].value_counts().plot(kind="bar", ax=axes[0, 0], color=["tomato", "steelblue"])
axes[0, 0].set_title("Survival Count")
axes[0, 0].set_xticklabels(["Did not survive", "Survived"], rotation=0)

# Survival by sex
df.groupby("Sex")["Survived"].mean().plot(kind="bar", ax=axes[0, 1], color=["steelblue", "tomato"])
axes[0, 1].set_title("Survival Rate by Sex")
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)

# Survival by class
df.groupby("Pclass")["Survived"].mean().plot(kind="bar", ax=axes[0, 2], color="mediumseagreen")
axes[0, 2].set_title("Survival Rate by Pclass")
axes[0, 2].set_xticklabels(axes[0, 2].get_xticklabels(), rotation=0)

# Age distribution
df["Age"].dropna().plot(kind="hist", bins=30, ax=axes[1, 0], color="steelblue", edgecolor="white")
axes[1, 0].set_title("Age Distribution")

# Fare distribution
df["Fare"].plot(kind="hist", bins=40, ax=axes[1, 1], color="mediumseagreen", edgecolor="white")
axes[1, 1].set_title("Fare Distribution")

# Survival by embarked
df.groupby("Embarked")["Survived"].mean().plot(kind="bar", ax=axes[1, 2], color="orchid")
axes[1, 2].set_title("Survival Rate by Embarked")
axes[1, 2].set_xticklabels(axes[1, 2].get_xticklabels(), rotation=0)

plt.suptitle("Titanic — Exploratory Data Analysis", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Data Cleaning and Feature Engineering ---

data = df.copy()

# Fill missing Age with median
data["Age"].fillna(data["Age"].median(), inplace=True)

# Fill missing Embarked with mode
data["Embarked"].fillna(data["Embarked"].mode()[0], inplace=True)

# Fill missing Fare with median
data["Fare"].fillna(data["Fare"].median(), inplace=True)

# Drop columns with too many missing values or low predictive value
data.drop(columns=[c for c in ["Cabin", "Ticket", "Name", "PassengerId"] if c in data.columns], inplace=True)

# Encode Sex and Embarked
data["Sex"] = data["Sex"].map({"male": 0, "female": 1})
data = pd.get_dummies(data, columns=["Embarked"], drop_first=True)

# Feature: family size
data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
data["IsAlone"] = (data["FamilySize"] == 1).astype(int)

print("Cleaned dataset shape:", data.shape)
print("Missing values remaining:", data.isnull().sum().sum())
data.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

X = data.drop(columns=["Survived"])
y = data["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

def evaluate(name, y_true, y_pred):
    print(f"\n{name}")
    print("-" * 40)
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score : {f1_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred, target_names=["Did not survive", "Survived"]))
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred), 4),
        "Recall": round(recall_score(y_true, y_pred), 4),
        "F1": round(f1_score(y_true, y_pred), 4)
    }

scores = []
print(f"Train: {X_train_sc.shape}, Test: {X_test_sc.shape}")

## Exercise 2: Decision Tree without Grid Search

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# max_depth=5 prevents overfitting on a small dataset.
# min_samples_split=10 avoids splits on very few samples.
# criterion='gini' is the standard impurity measure for classification.
dt = DecisionTreeClassifier(max_depth=5, min_samples_split=10, criterion="gini", random_state=42)
dt.fit(X_train_sc, y_train)
y_pred_dt = dt.predict(X_test_sc)

scores.append(evaluate("Decision Tree (no tuning)", y_test, y_pred_dt))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt="d",
            cmap="Blues", cbar=False,
            xticklabels=["0", "1"], yticklabels=["0", "1"])
plt.title("Decision Tree — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Exercise 3: Decision Tree with Grid Search

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_dt = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"]
}

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt, cv=5, scoring="f1", n_jobs=-1
)
grid_dt.fit(X_train_sc, y_train)

print(f"Best params: {grid_dt.best_params_}")
print(f"Best CV F1:  {grid_dt.best_score_:.4f}")

y_pred_dt_gs = grid_dt.best_estimator_.predict(X_test_sc)
scores.append(evaluate("Decision Tree + Grid Search", y_test, y_pred_dt_gs))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_dt_gs), annot=True, fmt="d",
            cmap="Blues", cbar=False,
            xticklabels=["0", "1"], yticklabels=["0", "1"])
plt.title("Decision Tree + Grid Search — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Exercise 4: KNN without Grid Search

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# n_neighbors=7: odd number avoids ties; sqrt(n_train) heuristic gives ~19, but
# smaller values work better on noisy Titanic data. metric='minkowski' with p=2
# is standard Euclidean distance.
knn = KNeighborsClassifier(n_neighbors=7, metric="minkowski", p=2)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)

scores.append(evaluate("KNN (no tuning)", y_test, y_pred_knn))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_knn), annot=True, fmt="d",
            cmap="Greens", cbar=False,
            xticklabels=["0", "1"], yticklabels=["0", "1"])
plt.title("KNN — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Exercise 5: KNN with Grid Search

In [ ]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "metric": ["minkowski", "manhattan"],
    "weights": ["uniform", "distance"]
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn, cv=5, scoring="f1", n_jobs=-1
)
grid_knn.fit(X_train_sc, y_train)

print(f"Best params: {grid_knn.best_params_}")
print(f"Best CV F1:  {grid_knn.best_score_:.4f}")

y_pred_knn_gs = grid_knn.best_estimator_.predict(X_test_sc)
scores.append(evaluate("KNN + Grid Search", y_test, y_pred_knn_gs))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_knn_gs), annot=True, fmt="d",
            cmap="Greens", cbar=False,
            xticklabels=["0", "1"], yticklabels=["0", "1"])
plt.title("KNN + Grid Search — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Exercise 6: Neural Network without Hyperparameter Tuning

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)

n_features = X_train_sc.shape[1]

nn = keras.Sequential([
    keras.layers.Input(shape=(n_features,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="sigmoid")
])

nn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

nn.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

history = nn.fit(
    X_train_sc, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

print(f"Training stopped at epoch {len(history.history['loss'])}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history.history["loss"], label="Train Loss")
axes[0].plot(history.history["val_loss"], label="Val Loss")
axes[0].set_title("Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train Accuracy")
axes[1].plot(history.history["val_accuracy"], label="Val Accuracy")
axes[1].set_title("Accuracy over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

y_pred_nn_prob = nn.predict(X_test_sc, verbose=0).flatten()
y_pred_nn = (y_pred_nn_prob >= 0.5).astype(int)

scores.append(evaluate("Neural Network (no tuning)", y_test, y_pred_nn))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_nn), annot=True, fmt="d",
            cmap="Purples", cbar=False,
            xticklabels=["0", "1"], yticklabels=["0", "1"])
plt.title("Neural Network — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Exercise 7 (Optional): Neural Network with Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scikeras.wrappers import KerasClassifier

def build_nn(n_units_1=64, n_units_2=32, dropout=0.3, learning_rate=0.001):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features,)),
        keras.layers.Dense(n_units_1, activation="relu"),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(n_units_2, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


keras_clf = KerasClassifier(
    model=build_nn,
    epochs=50,
    batch_size=32,
    verbose=0,
    random_state=42
)

param_dist_nn = {
    "model__n_units_1": [32, 64, 128],
    "model__n_units_2": [16, 32, 64],
    "model__dropout": [0.2, 0.3, 0.4],
    "model__learning_rate": [0.001, 0.005, 0.01],
    "batch_size": [16, 32]
}

rand_nn = RandomizedSearchCV(
    keras_clf,
    param_dist_nn,
    n_iter=10,
    cv=3,
    scoring="f1",
    n_jobs=1,
    random_state=42
)
rand_nn.fit(X_train_sc, y_train)

print(f"Best params: {rand_nn.best_params_}")
print(f"Best CV F1:  {rand_nn.best_score_:.4f}")

y_pred_nn_tuned = rand_nn.best_estimator_.predict(X_test_sc)
scores.append(evaluate("Neural Network + Randomized Search", y_test, y_pred_nn_tuned))

plt.figure(figsize=(4, 3))
sns.heatmap(confusion_matrix(y_test, y_pred_nn_tuned), annot=True, fmt="d",
            cmap="Purples", cbar=False,
            xticklabels=["0", "1"], yticklabels=["0", "1"])
plt.title("Neural Network + Tuning — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## Model Comparison Report

In [ ]:
summary = pd.DataFrame(scores).sort_values("F1", ascending=False).reset_index(drop=True)
print(summary.to_string(index=False))

metrics = ["Accuracy", "Precision", "Recall", "F1"]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

colors = ["steelblue", "tomato", "mediumseagreen", "orchid"]

for ax, metric, color in zip(axes, metrics, colors):
    ranked = summary.sort_values(metric, ascending=True)
    bars = ax.barh(ranked["Model"], ranked[metric], color=color)
    ax.set_xlim(0, 1)
    ax.set_title(metric)
    ax.set_xlabel(metric)
    for bar, val in zip(bars, ranked[metric]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=8)

plt.suptitle("Model Comparison — Titanic Survival Prediction", fontsize=13)
plt.tight_layout()
plt.show()

best = summary.iloc[0]
print(f"\nBest model by F1-Score: {best['Model']}")
print(f"  Accuracy : {best['Accuracy']}")
print(f"  Precision: {best['Precision']}")
print(f"  Recall   : {best['Recall']}")
print(f"  F1-Score : {best['F1']}")

## Feature Importance (Decision Tree)

In [ ]:
best_dt = grid_dt.best_estimator_
importances = pd.Series(best_dt.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(7, 5))
importances.plot(kind="barh", color="steelblue")
plt.title("Feature Importances — Best Decision Tree")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()